<a href="https://colab.research.google.com/github/mahamfurqan00-code/Lahore-City-Explorer-pandas-Folium-/blob/main/Lahore_City_Explorer_(pandas_%2B_Folium).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 — Lahore City Explorer (pandas + Folium)

**Solution notebook.** Every code cell is commented to explain the *why* behind each step.

File used: `lahore_places.csv`

> Note: the final map object is displayed inline (Folium maps render directly
> in a Jupyter cell), and is also saved to `lahore_map.html` so it can be
> opened outside the notebook too.

In [1]:
# Imports
import pandas as pd
import folium


## Part A — Load & Explore

In [2]:
from folium.plugins import HeatMap
df = pd.read_csv("lahore_places.csv")
print(df.shape)
df.head()

(20, 6)


,name,category,lat,lon,avg_rating,daily_footfall
0,Badshahi Mosque,Heritage,31.5881,74.3103,4.8,9500
1,Lahore Fort,Heritage,31.5882,74.3150,4.7,7200
2,Wazir Khan Mosque,Heritage,31.5828,74.3323,4.6,4100
3,Shalimar Gardens,Heritage,31.5952,74.3811,4.5,3800
4,Minar-e-Pakistan,Heritage,31.5925,74.3095,4.6,6100


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            20 non-null     object 
 1   category        20 non-null     object 
 2   lat             20 non-null     float64
 3   lon             20 non-null     float64
 4   avg_rating      20 non-null     float64
 5   daily_footfall  20 non-null     int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 1.1+ KB


In [4]:
# Average rating and TOTAL footfall per category
# (mean makes sense for rating; sum makes sense for footfall — we want total traffic, not average)
category_summary = df.groupby("category").agg(
    avg_rating=("avg_rating", "mean"),
    total_footfall=("daily_footfall", "sum"),
).sort_values("total_footfall", ascending=False)

category_summary

,avg_rating,total_footfall
category,,
Shopping,4.183333,51300
Heritage,4.650000,41700
Education,4.200000,15000
Park,4.175000,13000
Recreation,4.000000,4700
Sports,4.500000,2600
Museum,4.400000,1800


In [5]:
# Single busiest place overall
busiest = df.sort_values("daily_footfall", ascending=False).iloc[0]
print(f"Busiest place: {busiest['name']} ({busiest['category']}) "
      f"with {busiest['daily_footfall']:,} daily visitors")

df.sort_values("daily_footfall", ascending=False).head()

Busiest place: Punjab University (New Campus) (Education) with 15,000 daily visitors


,name,category,lat,lon,avg_rating,daily_footfall
18,Punjab University (New Campus),Education,31.4998,74.2953,4.2,15000
8,Emporium Mall,Shopping,31.4802,74.3439,4.3,12500
16,Data Darbar,Heritage,31.5852,74.3184,4.7,11000
9,Packages Mall,Shopping,31.4746,74.3625,4.4,10200
7,Anarkali Bazaar,Shopping,31.5686,74.3103,4.1,9900


## Part B — Base Map

In [6]:
# Center roughly on Lahore, with a zoom level that shows the whole city
m = folium.Map(location=[31.53, 74.34], zoom_start=12, tiles="OpenStreetMap")
m

## Part C — Color-Code by Category

In [7]:
# Check exactly which categories exist before assigning colors —
# don't guess, read it off the real data.
print(sorted(df["category"].unique()))

['Education', 'Heritage', 'Museum', 'Park', 'Recreation', 'Shopping', 'Sports']


In [8]:
# One distinct color per category
category_colors = {
    "Heritage": "darkred",
    "Museum": "purple",
    "Shopping": "blue",
    "Sports": "orange",
    "Park": "green",
    "Recreation": "cadetblue",
    "Education": "gray",
}

# Fallback color for any category not explicitly listed above
DEFAULT_COLOR = "black"


## Part D — Size by Footfall + Part F — Toggleable Layers (built together)

In [10]:
# Rebuild the map fresh so we don't stack duplicate layers if this cell re-runs.
m = folium.Map(location=[31.53, 74.34], zoom_start=12, tiles="OpenStreetMap")

# One FeatureGroup per category so students can toggle categories on/off later.
feature_groups = {
    category: folium.FeatureGroup(name=category)
    for category in df["category"].unique()
}

for row in df.itertuples(index=False):
    color = category_colors.get(row.category, DEFAULT_COLOR)

    # Radius scaled by footfall: divisor chosen so circles stay readable
    # (not tiny dots, not overlapping blobs) for this dataset's value range.
    radius = row.daily_footfall / 1500

    marker = folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=radius,
        popup=folium.Popup(
            f"<b>{row.name}</b><br>"
            f"Category: {row.category}<br>"
            f"Rating: {row.avg_rating}★<br>"
            f"Daily footfall: {row.daily_footfall:,}",
            max_width=250,
        ),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
    )
    marker.add_to(feature_groups[row.category])

# Add every category's FeatureGroup to the map
for group in feature_groups.values():
    group.add_to(m)

## Part E — Heatmap Layer

In [11]:
# Weighted heatmap using footfall as the weight, so it visualizes DENSITY
# separately from the individual colored markers above.
heat_data = df[["lat", "lon", "daily_footfall"]].values.tolist()
HeatMap(heat_data, name="Footfall Heatmap", radius=25, blur=15).add_to(m)

In [12]:
# Layer control lets a viewer toggle each category (and the heatmap) on/off
folium.LayerControl(collapsed=False).add_to(m)

m

## Part G — Save

In [13]:
m.save("lahore_map.html")
print("Saved lahore_map.html — open it in a browser to explore the interactive map.")

Saved lahore_map.html — open it in a browser to explore the interactive map.


## Bonus Challenge

- A walking route between nearby heritage sites, with straight-line distances printed
- A "must-visit only" filtered map (rating >= 4.5)

In [14]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    """Straight-line distance between two lat/lon points, in kilometers.
    This is a simplification (no real roads/routing) — just enough to
    illustrate the idea, matching a Dijkstra-style routing project's spirit.
    """
    R = 6371  # Earth's radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


route_names = ["Badshahi Mosque", "Lahore Fort", "Data Darbar"]
route_df = df[df["name"].isin(route_names)].set_index("name").loc[route_names]

route_map = folium.Map(location=[31.585, 74.315], zoom_start=14)
coords = list(zip(route_df["lat"], route_df["lon"]))

for name, (lat, lon) in zip(route_names, coords):
    folium.Marker([lat, lon], popup=name, icon=folium.Icon(color="darkred")).add_to(route_map)

folium.PolyLine(coords, color="blue", weight=4, opacity=0.8).add_to(route_map)

# Print distance between each consecutive stop
print("Route distances:")
for i in range(len(coords) - 1):
    d = haversine_km(*coords[i], *coords[i + 1])
    print(f"  {route_names[i]} -> {route_names[i+1]}: {d:.2f} km")

route_map

Route distances:
  Badshahi Mosque -> Lahore Fort: 0.45 km
  Lahore Fort -> Data Darbar: 0.46 km


In [15]:
# "Must-visit only" filtered map: places rated 4.5 or higher
top_rated = df[df["avg_rating"] >= 4.5]
print(f"{len(top_rated)} places meet the 4.5+ rating bar:")
print(top_rated[["name", "category", "avg_rating"]].sort_values("avg_rating", ascending=False))

must_visit_map = folium.Map(location=[31.53, 74.34], zoom_start=12)
for row in top_rated.itertuples(index=False):
    color = category_colors.get(row.category, DEFAULT_COLOR)
    folium.Marker(
        [row.lat, row.lon],
        popup=f"{row.name} ({row.avg_rating}★)",
        icon=folium.Icon(color="green" if row.avg_rating >= 4.7 else "orange"),
    ).add_to(must_visit_map)

must_visit_map

7 places meet the 4.5+ rating bar:
                 name  category  avg_rating
0     Badshahi Mosque  Heritage         4.8
1         Lahore Fort  Heritage         4.7
16        Data Darbar  Heritage         4.7
2   Wazir Khan Mosque  Heritage         4.6
4    Minar-e-Pakistan  Heritage         4.6
3    Shalimar Gardens  Heritage         4.5
10    Gaddafi Stadium    Sports         4.5
